# L4: Support Data Insight Analysis

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ <b>Note <code>(Kernel Starting)</code>:</b> This notebook takes about 30 seconds to be ready to use. You may start and watch the video while you wait.</p>

## Initial Imports

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

# Load environment variables
import sys
sys.path.insert(0, "/Users/karli/Desktop/crew-ai-agents")

# Load environment variables
from helper import load_env
load_env()

import os
import yaml
from crewai import LLM
from crewai import Agent, Task, Crew

gemini_llm = LLM(
    model="gemini/gemini-3.5-flash",
    max_retries=6,         # Aumenta los reintentos automáticos
    timeout=120
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


<p style="background-color:#fff6ff; padding:15px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px"> 💻 &nbsp; <b>Access <code>requirements.txt</code> and <code>helper.py</code> files:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Open"</em>. For more help, please see the <em>"Appendix - Tips and Help"</em> Lesson.</p>

## Loading Tasks and Agents YAML files

In [2]:
# Define file paths for YAML configurations
files = {
    'agents': 'config/agents.yaml',
    'tasks': 'config/tasks.yaml'
}

# Load configurations from YAML files
configs = {}
for config_type, file_path in files.items():
    with open(file_path, 'r') as file:
        configs[config_type] = yaml.safe_load(file)

# Assign loaded configurations to specific variables
agents_config = configs['agents']
tasks_config = configs['tasks']

## Using FileReadTool

In [3]:
from crewai_tools import FileReadTool
csv_tool = FileReadTool(file_path='./support_tickets_data.csv')

## Creating Agents, Tasks and Crew

In [4]:
# Creating Agents
suggestion_generation_agent = Agent(
  config=agents_config['suggestion_generation_agent'],
  tools=[csv_tool]
)

reporting_agent = Agent(
  config=agents_config['reporting_agent'],
  tools=[csv_tool]
)

chart_generation_agent = Agent(
  config=agents_config['chart_generation_agent'],
  allow_code_execution=False
)

# Creating Tasks
suggestion_generation = Task(
  config=tasks_config['suggestion_generation'],
  agent=suggestion_generation_agent
)

table_generation = Task(
  config=tasks_config['table_generation'],
  agent=reporting_agent
)

chart_generation = Task(
  config=tasks_config['chart_generation'],
  agent=chart_generation_agent
)

final_report_assembly = Task(
  config=tasks_config['final_report_assembly'],
  agent=reporting_agent,
  context=[suggestion_generation, table_generation, chart_generation]
)


# Creating Crew
support_report_crew = Crew(
  agents=[
    suggestion_generation_agent,
    reporting_agent,
    chart_generation_agent
  ],
  tasks=[
    suggestion_generation,
    table_generation,
    chart_generation,
    final_report_assembly
  ],
  verbose=True
)


## Testing our Crew

In [22]:
import asyncio
import os
# Eliminar la variable falsa para evitar que LiteLLM intente usar OpenAI
if "OPENAI_API_KEY" in os.environ:
    del os.environ["OPENAI_API_KEY"]
# Y luego ejecuta tu comando con normalidad:
#result = await asyncio.to_thread(support_report_crew.kickoff)
result = await support_report_crew.kickoff_async()
print(result)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.33                                                                                       │
│  Latest version:  1.15.41                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: c15a1959-4f69-40e1-9c14-09ac80def4c7                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Generate actionable suggestions for resolving each classified support ticket. The suggestions should be  │
│  based on: - Issue Type: Tailor suggestions to the specific type of issue reported. - Historical Data: Use      │
│  historical data such as resolution_time_minutes and                                                            │
│    satisfaction_rating to inform the suggestions.                                                               │
│  - Customer Feedback: Incorporate insights from customer_comments to                                            │
│    customize the suggestions further.                                                                           │
│                                                                                                                 │
│  The goal is to provide clear, actionable steps that the support team can take to resolve each issue            │
│  efficiently and effectively.                                                                                   │
│                                                                                                                 │
│  ID: e5f4116c-b3b6-4e48-bd00-ab2779a7fe41                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Suggestion Engine                                                                                       │
│                                                                                                                 │
│  Task: Generate actionable suggestions for resolving each classified support ticket. The suggestions should be  │
│  based on: - Issue Type: Tailor suggestions to the specific type of issue reported. - Historical Data: Use      │
│  historical data such as resolution_time_minutes and                                                            │
│    satisfaction_rating to inform the suggestions.                                                               │
│  - Customer Feedback: Incorporate insights from customer_comments to                                            │
│    customize the suggestions further.                                                                           │
│                                                                                                                 │
│  The goal is to provide clear, actionable steps that the support team can take to resolve each issue            │
│  efficiently and effectively.                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ERROR:root:OpenAI API call failed: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-dummy*********************tion. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}
ERROR:root:OpenAI API call failed: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-dummy*********************tion. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}


╭───────────────────────────────────────────────── ❌ LLM Error ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  LLM Call Failed                                                                                                │
│  Error: OpenAI API call failed: Error code: 401 - {'error': {'message': 'Incorrect API key provided:            │
│  sk-dummy*********************tion. You can find your API key at                                                │
│  https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key',    │
│  'param': None}, 'status': 401}                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (4 outputs left out)

An unknown error occurred. Please check the details below.
Error details: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-dummy*********************tion. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}


╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: Generate actionable suggestions for resolving each classified support ticket. The suggestions should be  │
│  based on: - Issue Type: Tailor suggestions to the specific type of issue reported. - Historical Data: Use      │
│  historical data such as resolution_time_minutes and                                                            │
│    satisfaction_rating to inform the suggestions.                                                               │
│  - Customer Feedback: Incorporate insights from customer_comments to                                            │
│    customize the suggestions further.                                                                           │
│                                                                                                                 │
│  The goal is to provide clear, actionable steps that the support team can take to resolve each issue            │
│  efficiently and effectively.                                                                                   │
│                                                                                                                 │
│  Agent: Suggestion Engine                                                                                       │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: c15a1959-4f69-40e1-9c14-09ac80def4c7                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-dummy*********************tion. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

## Training your crew and agents

In [ ]:
support_report_crew.train(n_iterations=1, filename='training.pkl')

## Comparing new test results

In [ ]:
support_report_crew.test(n_iterations=1, openai_model_name='gpt-4o')

In [ ]:
# Display the Trello screenshot
from IPython.display import Image, display

# Load and display the image
test_image = Image(filename='test_before_training.png', width=368)
display(test_image)

## Kicking off Crew

In [ ]:
result = support_report_crew.kickoff()

## Result

In [ ]:
from IPython.display import display, Markdown
display(Markdown(result.raw))